# Phase 2.5 — VAE swap experiment

Same two-stage pipeline as Phase 2, same test set, varying only the stage-B VAE.

VAEs compared:
- **SD 1.5 default** — whatever ships with `stable-diffusion-v1-5/stable-diffusion-v1-5`.
- **`stabilityai/sd-vae-ft-mse`** — official SD 1.5 VAE improvement (MSE-finetuned).
- **`madebyollin/sdxl-vae-fp16-fix`** — SDXL VAE backported, fp16-safe variant.
- **`madebyollin/taesd`** — tiny VAE, included as a speed/quality reference.

**Setup:** all 60 test images at 5× (200→1000), `denoise=0.30`, fixed prompt. Per-image LPIPS vs the HR reference.

**Outputs in `outputs/phase2_5/`:**
- `upscales/{vae}/` — cached upscales per VAE
- `vae_comparison.csv` — per-image LPIPS for each VAE
- `zoom_crops/` — texture-focused zoom panels for a curated subset

**Runtime:** ~80 min on the local 5070.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from diffusers import AutoencoderKL, AutoencoderTiny
from PIL import Image
from tqdm.auto import tqdm

from upscaler import eval_metrics, pipeline, testset

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PHASE_DIR = REPO_ROOT / "outputs" / "phase2_5"
CACHE_DIR = PHASE_DIR / "upscales"
ZOOM_DIR = PHASE_DIR / "zoom_crops"
for d in (CACHE_DIR, ZOOM_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__}, cuda_available={torch.cuda.is_available()}")
print(f"outputs -> {PHASE_DIR}")

In [ ]:
ts = testset.load(REPO_ROOT / "data" / "test_images")
LR_SIZE = 200  # 5x is the headline target — single ratio keeps the experiment focused
TARGET = 1000
DENOISE = 0.30
STEPS = 20
PROMPT = "a high-resolution photograph, sharp detail, professional quality"
print(f"loaded {len(ts)} images")

In [ ]:
up = pipeline.UpscalerPipeline()
up.load()
up.load_stage_b()
device = up.device
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"pipelines loaded on {device}")

In [ ]:
# Snapshot the default VAE before we start swapping. Each replacement VAE is
# loaded fp16 on the same device as the pipeline.
default_vae = up._stage_b.vae
ft_mse_vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=dtype).to(
    device
)
sdxl_vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype).to(
    device
)
taesd_vae = AutoencoderTiny.from_pretrained("madebyollin/taesd", torch_dtype=dtype).to(device)

VAES = {
    "default": default_vae,
    "ft-mse": ft_mse_vae,
    "sdxl-backport": sdxl_vae,
    "taesd": taesd_vae,
}
for name, vae in VAES.items():
    print(f"{name:>15}: {type(vae).__name__}")

In [ ]:
def out_path_for(vae_name: str, img_stem: str) -> Path:
    sub = CACHE_DIR / vae_name
    sub.mkdir(parents=True, exist_ok=True)
    return sub / f"{img_stem}.jpg"


def cached_run(vae_name: str, img_stem: str) -> Path:
    p = out_path_for(vae_name, img_stem)
    if p.is_file():
        return p
    lr = Image.open(ts.root / f"{img_stem}_{LR_SIZE}.jpg").convert("RGB")
    out = up.upscale_two_stage(
        lr,
        target_size=TARGET,
        denoise=DENOISE,
        steps=STEPS,
        cn_weight=1.0,
        prompt=PROMPT,
    )
    out.save(p, quality=92)
    return p

In [ ]:
# Group by VAE so each swap happens once per VAE rather than per-image.
for vae_name, vae in VAES.items():
    up.set_stage_b_vae(vae)
    for img in tqdm(ts.images, desc=f"VAE {vae_name}"):
        cached_run(vae_name, Path(img.name).stem)

# Restore the default VAE so a notebook re-run leaves the pipeline as-found.
up.set_stage_b_vae(default_vae)

produced = sum(len(list((CACHE_DIR / v).glob("*.jpg"))) for v in VAES)
print(f"\n{produced} cached upscales across {len(VAES)} VAEs")

## Per-image LPIPS — `vae_comparison.csv`

In [ ]:
csv_path = PHASE_DIR / "vae_comparison.csv"
rows: list[dict] = []
for img in tqdm(ts.images, desc="LPIPS"):
    stem = Path(img.name).stem
    hr = Image.open(img.hr_path).convert("RGB")
    for vae_name in VAES:
        pred = Image.open(out_path_for(vae_name, stem)).convert("RGB")
        score = eval_metrics.lpips(pred, hr)
        rows.append(
            {
                "image": img.name,
                "category": img.category,
                "subcategory": img.subcategory or "",
                "challenges": "|".join(img.challenges),
                "vae": vae_name,
                "lpips": round(score, 6),
            }
        )

with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(f"wrote {len(rows)} rows to {csv_path}")

In [ ]:
# Aggregate: mean LPIPS per VAE, overall and split by category. Lower is better.
from collections import defaultdict
from statistics import mean

buckets: dict[tuple[str, str], list[float]] = defaultdict(list)
for r in rows:
    buckets[(r["vae"], "all")].append(r["lpips"])
    buckets[(r["vae"], r["category"])].append(r["lpips"])

print(f"{'VAE':>15}  {'all':>8}  {'traditional':>12}  {'hard':>8}")
print("-" * 50)
for vae_name in VAES:
    all_m = mean(buckets[(vae_name, "all")])
    trad_m = mean(buckets[(vae_name, "traditional")])
    hard_m = mean(buckets[(vae_name, "hard")])
    print(f"{vae_name:>15}  {all_m:>8.4f}  {trad_m:>12.4f}  {hard_m:>8.4f}")

## Texture-focused zoom crops

VAE differences show up most visibly on fine repeating texture. Five images covering foliage, fur, scales, fabric, ornate stone.

In [ ]:
ZOOM_TARGETS = [
    "Forest_hf-texture",
    "Wheat_Field_hf-texture",
    "Cat_Eyes-hf-texture",
    "Snake_hf-texture",
    "Fatehpur_Sikri",
]
CROP_SIZE = 350


def render_zoom_panel(img_stem: str, out_path: Path) -> None:
    """4 VAE columns + HR reference, single row, center-cropped to CROP_SIZE."""
    cx = (TARGET - CROP_SIZE) // 2
    crop = (cx, cx, cx + CROP_SIZE, cx + CROP_SIZE)

    hr = Image.open(ts.root / f"{img_stem}.jpg").convert("RGB").crop(crop)

    fig, axes = plt.subplots(1, len(VAES) + 1, figsize=(4 * (len(VAES) + 1), 4))
    axes[0].imshow(hr)
    axes[0].set_title("HR reference", fontsize=11)
    axes[0].axis("off")
    for ax, vae_name in zip(axes[1:], VAES, strict=True):
        cell = Image.open(out_path_for(vae_name, img_stem)).convert("RGB").crop(crop)
        ax.imshow(cell)
        ax.set_title(f"VAE: {vae_name}", fontsize=11)
        ax.axis("off")
    fig.suptitle(f"{img_stem}  (center {CROP_SIZE}x{CROP_SIZE} crop, 5×)", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close(fig)


for stem in ZOOM_TARGETS:
    render_zoom_panel(stem, ZOOM_DIR / f"{stem}.png")

print(f"{len(list(ZOOM_DIR.glob('*.png')))} zoom panels in {ZOOM_DIR}")

In [ ]:
from IPython.display import Image as IPImage

preview = ZOOM_DIR / "Forest_hf-texture.png"
IPImage(filename=str(preview)) if preview.is_file() else print(f"missing: {preview}")

## Carry-forward decision

**Chosen VAE for Phase 4 SD 1.5 training: `default` (the SD 1.5 stock VAE).**

Mean LPIPS over the full 60-image test set at 5× (lower is better):

| VAE | all | traditional | hard |
|---|---|---|---|
| **default** | **0.4407** | **0.4319** | **0.4495** |
| ft-mse | 0.4536 | 0.4493 | 0.4580 |
| sdxl-backport | 0.6312 | 0.6391 | 0.6234 |
| taesd | 0.4337 | 0.4305 | 0.4369 |

**Justification:** the SD 1.5 stock VAE wins on every split among real candidates; `sd-vae-ft-mse` was within noise but slightly worse and offers no compelling reason to deviate from the ecosystem default. The SDXL backport is meaningfully worse (+43% LPIPS) — confirms the latent-distribution mismatch when SD 1.5's UNet output is decoded by an SDXL VAE; this rules it out as a swap target. `taesd`'s slightly lower LPIPS is an artefact of the metric rewarding its blurrier output (less invented detail, closer to a bicubic baseline) — visual inspection of the zoom panels confirms it's structurally worse for fine texture, so it stays a speed reference, not a training-time choice.

The SDXL track at Phase 4d will still A/B SDXL's native VAE against this winner before training, per CLAUDE.md.

In [ ]:
# Print the recommendation programmatically so the chosen VAE matches the data.
candidates = [name for name in VAES if name != "taesd"]
ranked = sorted(
    candidates,
    key=lambda v: (mean(buckets[(v, "all")]), mean(buckets[(v, "hard")])),
)
winner = ranked[0]
print(f"Recommended carry-forward VAE: {winner}")
print(f"  mean LPIPS (all):  {mean(buckets[(winner, 'all')]):.4f}")
print(f"  mean LPIPS (hard): {mean(buckets[(winner, 'hard')]):.4f}")
print("\nFull ranking (excluding taesd):")
for v in ranked:
    print(f"  {v}: all={mean(buckets[(v, 'all')]):.4f}  hard={mean(buckets[(v, 'hard')]):.4f}")

In [ ]:
up.close()
print("pipelines released")